## 1. Setup: Authenticate with Google and Initialize APIs

To interact with Google Docs and Google Sheets, we need to authenticate and enable their respective APIs.

In [ ]:
# Install necessary libraries
# !pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build

# Authenticate to Colab with your Google account
auth.authenticate_user()

# Build the Docs, Sheets, and Drive API services
docs_service = build('docs', 'v1')
sheets_service = build('sheets', 'v4')
drive_service = build('drive', 'v3') # Initialize Drive API

print("Google Docs, Sheets, and Drive API services initialized.")

## 2. Get Google Doc Content (Example Input)

Before processing, you need to get the content of your Google Doc. Here's a placeholder for where you would load your document content. For this example, I'll use the sample transcript you provided.

### How to get a Google Doc document ID
1. You can find the document ID directly in the Google Doc's URL. It's the string of characters between `/d/` and `/edit` (or `/view`, `/present`, etc.) in the URL.

In [1]:
# Replace this with actual content fetched from a Google Doc using the Docs API
# For example: document_id = 'YOUR_DOCUMENT_ID'
#document_id = 'YOUR_DOCUMENT_ID'
#document = docs_service.documents().get(documentId=document_id).execute()
#doc_content = ''
#for content in document.get('body').get('content'):
#    if 'paragraph' in content:
#        elements = content.get('paragraph').get('elements')
#        for elem in elements:
#            if 'textRun' in elem:
#                doc_content += elem.get('textRun').get('content')

#print(doc_content)

# Sample transcript content
sample_transcript_content = """
1
00:00:00.070 --> 00:00:01.070
John Doe: Sounds good.

2
00:00:02.530 --> 00:00:07.360
Jane Doe: Oh, thank goodness that time it worked. Okay, so… Now.
"""

print(sample_transcript_content)


1
00:00:00.070 --> 00:00:01.070
John Doe: Sounds good.

2
00:00:02.530 --> 00:00:07.360
Jane Doe: Oh, thank goodness that time it worked. Okay, so… Now.



## 3. Parse the Transcript Content

This function will take the raw transcript text and parse it into a structured format (a list of dictionaries), separating the line number, timestamps, speaker, and utterance text into individual fields.

In [2]:
import re

def parse_transcript(transcript_text):
    utterances = []
    # Split the transcript into blocks, each block representing an utterance
    # We're looking for patterns like:\nNUMBER\nTIMESTAMP\nSPEAKER: TEXT
    blocks = re.split(r'\n(?=\d+\n\d{2}:\d{2}:\d{2}\.\d{3})', transcript_text.strip())

    for block in blocks:
        lines = block.strip().split('\n')
        if len(lines) >= 3:
            try:
                line_number = int(lines[0])
                timestamp_str = lines[1]
                speaker_utterance_str = lines[2]

                # Parse timestamp
                time_match = re.match(r'(\d{2}:\d{2}:\d{2}\.\d{3}) --> (\d{2}:\d{2}:\d{2}\.\d{3})', timestamp_str)
                start_time = time_match.group(1) if time_match else ''
                end_time = time_match.group(2) if time_match else ''

                # Separate speaker and utterance
                speaker_match = re.match(r'([^:]+):\s*(.*)', speaker_utterance_str)
                speaker = speaker_match.group(1).strip() if speaker_match else 'Unknown Speaker'
                utterance_text = speaker_match.group(2).strip() if speaker_match else speaker_utterance_str.strip()

                utterances.append({
                    'Line Number': line_number,
                    'Start Time': start_time,
                    'End Time': end_time,
                    'Speaker': speaker,
                    'Utterance': utterance_text
                })
            except Exception as e:
                print(f"Could not parse block: {block}\nError: {e}")

    return utterances

# Parse the sample transcript
#parsed_data = parse_transcript(doc_content)
parsed_data = parse_transcript(sample_transcript_content)

# Display the parsed data
import pandas as pd
df = pd.DataFrame(parsed_data)
display(df)

,Line Number,Start Time,End Time,Speaker,Utterance
0,1,00:00:00.070,00:00:01.070,John Doe,Sounds good.
1,2,00:00:02.530,00:00:07.360,Jane Doe,"Oh, thank goodness that time it worked. Okay, ..."


## 4. Write to Google Sheet

Now, we'll take the parsed data (DataFrame) and write it to a new Google Sheet. You'll need to provide a title for your new spreadsheet.

### How to get a Google Drive Folder ID:

1.  **Navigate to the folder** in Google Drive where your notebook is located (e.g., 'Colab Notebooks' or a custom folder).
2.  Look at the **URL in your browser's address bar**. It will look something like this:
    `https://drive.google.com/drive/folders/YOUR_FOLDER_ID_HERE`
3.  **Copy the `YOUR_FOLDER_ID_HERE` part** of the URL. This is the folder ID you'll use below.

In [ ]:
# Replace 'YOUR_FOLDER_ID_HERE' with the actual folder ID you copied from Google Drive
# Example: If your notebook is in 'My Drive/Colab Notebooks', find the folder ID for 'Colab Notebooks'.
drive_folder_id = 'YOUR_FOLDER_ID_HERE' # <--- UPDATE THIS WITH YOUR FOLDER ID

def write_to_google_sheet_in_folder(dataframe, spreadsheet_title='Interview Transcripts', folder_id=None):
    # Create a new spreadsheet
    spreadsheet_body = {
        'properties': {
            'title': spreadsheet_title
        }
    }

    # Create the spreadsheet first (it will be created in My Drive by default)
    spreadsheet = sheets_service.spreadsheets().create(body=spreadsheet_body,
                                        fields='spreadsheetId').execute()
    spreadsheet_id = spreadsheet.get('spreadsheetId')

    if folder_id:
        # Move the newly created spreadsheet to the specified folder using Drive API
        file_metadata = {
            'parents': [folder_id]
        }
        # Retrieve the existing parents to remove them
        file = drive_service.files().get(fileId=spreadsheet_id, fields='parents').execute()
        previous_parents = ",".join(file.get('parents'))

        drive_service.files().update(
            fileId=spreadsheet_id,
            addParents=folder_id,
            removeParents=previous_parents,
            fields='id, parents'
        ).execute()
        print(f"Spreadsheet moved to folder ID: {folder_id}")

    print(f"Spreadsheet ID: {spreadsheet_id}")
    print(f"Spreadsheet URL: https://docs.google.com/spreadsheets/d/{spreadsheet_id}")

    # Prepare data for Sheets API
    # Convert DataFrame to a list of lists, including header
    values = [dataframe.columns.values.tolist()] + dataframe.values.tolist()

    body = {
        'values': values
    }
    result = sheets_service.spreadsheets().values().update(
        spreadsheetId=spreadsheet_id,
        range='Sheet1!A1',
        valueInputOption='RAW',
        body=body).execute()

    print(f"{result.get('updatedCells')} cells updated.")
    return spreadsheet_id

# Call the updated function with your folder ID
# Make sure to replace 'YOUR_FOLDER_ID_HERE' with your actual folder ID above.
# If you don't provide a folder_id, it will create it in your My Drive.
# For example, if you want it in the same folder as this Colab notebook, find that folder's ID.
new_sheet_id = write_to_google_sheet_in_folder(df, spreadsheet_title='Interview Transcript Data - In Folder', folder_id=drive_folder_id)
print(f"New Sheet created with ID: {new_sheet_id}")